# EDTok: audit of the complete downloaded public repository

[Paper](https://arxiv.org/html/2505.02250) · [Official GitHub repository](https://github.com/cbickham3232/EDTok-A-Dataset-for-Eating-Disorder-Content-on-TikTok). The complete repository is now present locally. The authors' public README states that only video IDs are distributed to comply with TikTok's terms; the paper's comments, creator metadata, and raw media are **not** in the downloaded repository. This notebook checks exactly what can be reproduced from that public release. Run it from the repository root or its `Oct 9` folder.

## 1. Downloaded repository inventory

**Purpose:** Verify the actual files in the newly downloaded EDTok repository, including scripts and the sample ID file.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'downloaded_datasets').exists() and (ROOT.parent / 'downloaded_datasets').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'downloaded_datasets' / 'EDTok-A-Dataset-for-Eating-Disorder-Content-on-TikTok-main'
required = [DATA / 'README.md', DATA / 'data' / 'metadata.csv',
            DATA / 'data' / 'sample_metadata.csv',
            DATA / 'scripts' / 'metadata_collection.py',
            DATA / 'scripts' / 'download_videos.py']
display(pd.DataFrame([{'file': str(p.relative_to(DATA)), 'present': p.is_file(),
                       'size_KB': round(p.stat().st_size / 1e3, 2) if p.is_file() else None}
                      for p in required]))
if not all(p.is_file() for p in required):
    raise FileNotFoundError('One or more expected public-repository files are missing.')

,file,present,size_KB
0,README.md,True,1.28
1,data/metadata.csv,True,860.80
2,data/sample_metadata.csv,True,1.00
3,scripts/metadata_collection.py,True,5.72
4,scripts/download_videos.py,True,4.79


**What this shows:** This is the full downloaded public repository, not just a copied CSV. The Python scripts describe how to retrieve data under the appropriate access conditions; their presence does not mean they contain the authors' comments or videos.

## 2. Full ID list versus sample ID list

**Purpose:** Check both CSV schemas, uniqueness, and whether the sample contains any new videos.

In [2]:
ids = pd.read_csv(DATA / 'data' / 'metadata.csv', dtype=str)
sample = pd.read_csv(DATA / 'data' / 'sample_metadata.csv', dtype=str)
old = pd.read_csv(ROOT / 'downloaded_datasets' / 'edtok' / 'metadata.csv', dtype=str)
display(pd.DataFrame([
    {'file': 'data/metadata.csv', 'rows': len(ids), 'columns': ', '.join(ids.columns),
     'unique_ids': ids.id.nunique(), 'blank_ids': int(ids.id.isna().sum())},
    {'file': 'data/sample_metadata.csv', 'rows': len(sample), 'columns': ', '.join(sample.columns),
     'unique_ids': sample.id.nunique(), 'blank_ids': int(sample.id.isna().sum())},
]))
print('Sample IDs also in full list:', len(set(sample.id) & set(ids.id)), 'out of', len(sample))
print('Full-list IDs identical to earlier local copy:', set(ids.id) == set(old.id))

,file,rows,columns,unique_ids,blank_ids
0,data/metadata.csv,43040,id,43040,0
1,data/sample_metadata.csv,50,id,50,0


Sample IDs also in full list: 50 out of 50
Full-list IDs identical to earlier local copy: True


**What this shows:** The full file contains 43,040 unique video IDs and the 50-row sample contains only IDs already in that full list. The new repository does **not** add content, comment rows, creator fields, or engagement metadata. The full ID set is identical to the earlier local copy.

## 3. ID format and paper-count check

**Purpose:** Compare the public ID total with the paper's final-video count; do not infer inaccessible attributes from IDs.

In [3]:
clean = ids.id.fillna('').str.strip()
display(pd.DataFrame([{
    'public_rows': len(ids), 'unique_nonblank_ids': clean[clean.ne('')].nunique(),
    'paper_final_videos': 43040, 'difference_from_paper': len(ids) - 43040,
    'blank_ids': int(clean.eq('').sum()),
    'extra_duplicate_rows': int(clean[clean.ne('')].duplicated().sum()),
    'non_numeric_ids': int((~clean.str.fullmatch(r'\d+')).sum()),
}]))
display(clean.str.len().value_counts().sort_index().rename_axis('id_length').to_frame('rows'))

,public_rows,unique_nonblank_ids,paper_final_videos,difference_from_paper,blank_ids,extra_duplicate_rows,non_numeric_ids
0,43040,43040,43040,0,0,0,0


,rows
id_length,
19,43040


**What this shows:** The 43,040 released IDs are distinct, nonblank, and 19 digits long, matching the paper's final-video count. An ID is a pointer, not a comment count or a claim that the post is still available today.

## 4. What the repository does not contain

**Purpose:** Test whether substantive statistics can be calculated from its actual released columns.

In [4]:
requirements = {
    'Per-video comment mean and median': ['comment_count', 'comments'],
    'Unique creator count': ['author_id', 'username'],
    'Publication-month distribution': ['create_time', 'publish_time'],
    'Individual comment themes': ['comment_text', 'text'],
    'Views and likes': ['view_count', 'like_count'],
}
display(pd.DataFrame([{'analysis': name,
                       'needed_field_in_public_full_csv': any(col in ids.columns for col in columns)}
                      for name, columns in requirements.items()]))
print('Paper-reported: 10,897 users, 388,332 top-level comments and 188,739 replies.')
print('Locally reproducible from released CSVs: 43,040 distinct video IDs only.')

,analysis,needed_field_in_public_full_csv
0,Per-video comment mean and median,False
1,Unique creator count,False
2,Publication-month distribution,False
3,Individual comment themes,False
4,Views and likes,False


Paper-reported: 10,897 users, 388,332 top-level comments and 188,739 replies.
Locally reproducible from released CSVs: 43,040 distinct video IDs only.


**What this shows:** All five substantive analyses remain unavailable from the downloaded repository. The paper's 577,071 comments-plus-replies total is a **paper-reported** number, not locally downloaded comment rows. Do not divide it by 43,040 and call the result an observed per-video mean.

## 5. What the scripts would require

**Purpose:** State which files are scripts and what extra access or retrieval would be needed for richer data.

In [5]:
readme = (DATA / 'README.md').read_text(encoding='utf-8')
print('README says only video IDs are publicly released:',
      'only publicly releasing the TikTok Video IDs' in readme)
print('README mentions TikTok Research API:', 'TikTok Research API' in readme)
print('Local comment-row files:', [p.name for p in DATA.rglob('*')
                                  if p.is_file() and 'comment' in p.name.lower()])
print('Local raw video files:', [p.name for p in DATA.rglob('*')
                                if p.is_file() and p.suffix.lower() in {'.mp4', '.mov', '.webm'}])

README says only video IDs are publicly released: True
README mentions TikTok Research API: True
Local comment-row files: []
Local raw video files: []


**What this shows:** The README confirms an ID-only public release and says TikTok Research API access is required to retrieve metadata. There are no comment datasets or raw video files in this folder. Running the scripts would be a **new collection**, with possible access loss and platform-policy requirements, not analysis of already downloaded data.

## Findings and limits

- The complete local EDTok repository contains **43,040 distinct video IDs** in `data/metadata.csv`, matching the paper's final-video count. `sample_metadata.csv` contains **50 IDs**, all already in the full list.
- The public repository adds scripts and documentation, **not** the authors' creator metadata, comment rows, engagement measurements, or raw videos. The earlier local ID copy has the same 43,040 IDs.
- The paper reports **10,897 users**, **388,332 top-level comments**, and **188,739 replies**. Those figures cannot be independently verified from this ID-only release. Per-video comment mean and median are **not calculated**.
- Recollection would need TikTok Research API access and a new retrieval date. It might not reproduce the paper's original corpus because availability and engagement can change.